# Notebook 01: Traditional Machine Learning Baselines

Goal of this notebook: Establish strong baseline performances using traditional Machine Learning algorithms (Ridge Classifier, Random Forest) on representations generated by the Moirai foundation model.

In this approach, Moirai is used strictly as a frozen feature extractor:
1. Raw time series are passed through Moirai to extract 3D contextual embeddings (Batch, Variables/Time, Features).
2. We apply various Pooling strategies (flatten, mean, max) over patches or variables.
3. These flattened vectors are then used to train standard Scikit-Learn classifiers.

In [2]:
import numpy as np

from tslearn.datasets import UCR_UEA_datasets
import torch
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from uni2ts.model.moirai import MoiraiModule
from sklearn.linear_model import RidgeClassifierCV
from sklearn.ensemble import RandomForestClassifier
from encoder import MoiraiEncoder
from utils import *

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PATCH_SIZES = [8, 16, 32, 64]

SIZE = "small"
DEVICE = "cuda"
NUM_VARS = 6

DO_MASK = True
KEEP_MASK_EMBEDDING = True

In [4]:
alphas_to_test = [0.1,1,10,30,100,300,1000,10000]

models = {
    'Ridge': RidgeClassifierCV(alphas=alphas_to_test, cv=3),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

## 1. Data Loading and Embedding Extraction

We pre-extract the representations for all the patch sizes we are interested in (64, 32, 16, and 8). This way, the raw data only passes through the heavy Moirai encoder once per scale.

In [5]:
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

n_classes = len(set(y_train))
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

NUM_CLASSES = len(torch.unique(y_train_tensor))

In [6]:
X_train_target_tensor, X_train_is_target_observed, X_train_is_target_padded = (
    preprocess_data(X_train, device=DEVICE)
)

X_test_target_tensor, X_test_is_target_observed, X_test_is_target_padded = (
    preprocess_data(X_test, device=DEVICE)
)
y_train_tensor = torch.tensor(y_train, dtype=torch.long, device=DEVICE)
y_test_tensor = torch.tensor(y_test, dtype=torch.long, device=DEVICE)

num_classes = len(torch.unique(y_train_tensor))
NUM_VARS = 6

print(X_train_target_tensor.shape)
print(X_train_is_target_observed.shape)
print(X_train_is_target_padded.shape)

torch.Size([2459, 36, 6])
torch.Size([2459, 36, 6])
torch.Size([2459, 36])


In [7]:
Z_train_patch = {}
Z_test_patch = {}
Z_train_mask_patch = {}
Z_test_mask_patch = {}

for patch in PATCH_SIZES:
    MODEL_PARAMS = {
        "patch_size": patch,
        "num_samples": 100,
        "target_dim": 6,
        "feat_dynamic_real_dim": 0,
        "past_feat_dynamic_real_dim": 0,
    }
    if DO_MASK == True:
        prediction_length = patch
    else:
        prediction_length = 0
    encoder = MoiraiEncoder(
        module=MoiraiModule.from_pretrained(f"Salesforce/moirai-1.1-R-{SIZE}"),
        prediction_length=prediction_length,
        context_length=36,
        **MODEL_PARAMS,
    )
    encoder.eval()
    encoder.to(DEVICE)
    
    with torch.no_grad():
        Z_train = encoder(
            past_target=X_train_target_tensor,
            past_observed_target=X_train_is_target_observed,
            past_is_pad=X_train_is_target_padded,
        )
        Z_test = encoder(
            past_target=X_test_target_tensor,
            past_observed_target=X_test_is_target_observed,
            past_is_pad=X_test_is_target_padded,
        )

        
    if DO_MASK:
        N, S, F = Z_train.shape
        P = S // NUM_VARS  
        
        
        Z_train_reshaped = Z_train.view(N, NUM_VARS, P, F)
        Z_test_reshaped = Z_test.view(Z_test.shape[0], NUM_VARS, P, F)

        Z_train_mask_only = Z_train_reshaped[:, :, -1:, :]
        Z_test_mask_only = Z_test_reshaped[:, :, -1:, :]

        if not KEEP_MASK_EMBEDDING:
            Z_train_reshaped = Z_train_reshaped[:, :, :-1, :]
            Z_test_reshaped = Z_test_reshaped[:, :, :-1, :]
        
        Z_train = Z_train_reshaped.reshape(N, -1, F)
        Z_test = Z_test_reshaped.reshape(Z_test.shape[0], -1, F)

    Z_train_mask_patch[patch] = Z_train_mask_only.reshape(N, -1, F)
    Z_test_mask_patch[patch] = Z_test_mask_only.reshape(Z_test.shape[0], -1, F)

    Z_train_patch[patch] = Z_train
    Z_test_patch[patch] = Z_test

OutOfMemoryError: CUDA out of memory. Tried to allocate 130.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 85.75 MiB is free. Process 38140 has 850.00 MiB memory in use. Process 41952 has 12.89 GiB memory in use. Including non-PyTorch memory, this process has 778.00 MiB memory in use. Of the allocated memory 558.40 MiB is allocated by PyTorch, and 93.60 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:

print(f"{'Patch Size':<12} | {'Train Shape':<25} | {'Test Shape':<25}")
for patch in PATCH_SIZES:
    train_tensor = Z_train_patch[patch]
    test_tensor = Z_test_patch[patch]
    print(f"{patch:<12} | {str(list(train_tensor.shape)):<25} | {str(list(test_tensor.shape)):<25}")

Patch Size   | Train Shape               | Test Shape               
8            | [2459, 36, 384]           | [2466, 36, 384]          
16           | [2459, 24, 384]           | [2466, 24, 384]          
32           | [2459, 18, 384]           | [2466, 18, 384]          
64           | [2459, 12, 384]           | [2466, 12, 384]          


## "Mask Only" Evaluation


Idea: In its pre-training objective, Moirai is a forecasting model. It looks at the historical context to predict the future, which is represented by a final Mask Patch. 

In [ ]:
mask_pooling_methods = [
    "flatten", 
    "mean_over_variables", 
    "max_over_variables", 
    "min_over_variables",
    "mean_max_min_over_variables"
]

df_mask_ridge = pd.DataFrame(index=mask_pooling_methods, columns=PATCH_SIZES)
df_mask_rf = pd.DataFrame(index=mask_pooling_methods, columns=PATCH_SIZES)
df_mask_ridge.index.name = "Pooling (Mask Only)"
df_mask_rf.index.name = "Pooling (Mask Only)"

for patch in PATCH_SIZES:
    Z_train = Z_train_mask_patch[patch]
    Z_test = Z_test_mask_patch[patch]
    
    for pool in mask_pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool, num_vars=NUM_VARS).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool, num_vars=NUM_VARS).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        clf_ridge = models['Ridge']
        clf_ridge.fit(X_train_scaled, y_train)
        df_mask_ridge.loc[pool, patch] = round(accuracy_score(y_test, clf_ridge.predict(X_test_scaled)), 3)
        
        clf_rf = models['RandomForest']
        clf_rf.fit(X_train_scaled, y_train)
        df_mask_rf.loc[pool, patch] = round(accuracy_score(y_test, clf_rf.predict(X_test_scaled)), 3)

In [ ]:
display(df_mask_ridge.astype(float))

,8,16,32,64
Pooling (Mask Only),,,,
flatten,0.588,0.591,0.601,0.601
mean_over_variables,0.567,0.564,0.571,0.577
max_over_variables,0.524,0.540,0.541,0.567
min_over_variables,0.535,0.537,0.536,0.564
mean_max_min_over_variables,0.568,0.567,0.565,0.583


In [ ]:
display(df_mask_rf.astype(float))

,8,16,32,64
Pooling (Mask Only),,,,
flatten,0.521,0.524,0.536,0.567
mean_over_variables,0.537,0.530,0.534,0.560
max_over_variables,0.508,0.505,0.501,0.555
min_over_variables,0.498,0.505,0.511,0.549
mean_max_min_over_variables,0.536,0.526,0.530,0.549


## Multiscale

In [ ]:
pool_method = "flatten"

X_train_multi_mask = []
X_test_multi_mask = []

for patch in PATCH_SIZES:
    Z_train = Z_train_mask_patch[patch]
    Z_test = Z_test_mask_patch[patch]
    
    X_train_pool = apply_pooling_pt(Z_train, pool_method, num_vars=NUM_VARS).cpu().numpy()
    X_test_pool = apply_pooling_pt(Z_test, pool_method, num_vars=NUM_VARS).cpu().numpy()
    
    X_train_multi_mask.append(X_train_pool)
    X_test_multi_mask.append(X_test_pool)

X_train_combined = np.concatenate(X_train_multi_mask, axis=1)
X_test_combined = np.concatenate(X_test_multi_mask, axis=1)

scaler = StandardScaler()
X_train_combined_scaled = scaler.fit_transform(X_train_combined)
X_test_combined_scaled = scaler.transform(X_test_combined)



results = {}
for model_name, clf in models.items():
    clf.fit(X_train_combined_scaled, y_train)
    y_pred = clf.predict(X_test_combined_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[model_name] = round(acc, 3)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.50345e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.74556e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=4.60111e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)


In [ ]:
for model_name, acc in results.items():
    print(f"Modèle : {model_name:<15} | Test Accuracy : {acc:.3f}")

Modèle : Ridge           | Test Accuracy : 0.633
Modèle : RandomForest    | Test Accuracy : 0.573


## 2. Single-Scale Evaluation and Pooling Strategies


Here, we evaluate the patch sizes one by one (64, then 32, etc.). Since our classifiers expect 2D vectors, we test different ways to aggregate the dimensions of our embeddings:

* **`flatten`**: Flattens everything. Retains all information but creates massive vectors (highly prone to overfitting).
* **`global_*`**: Averages or takes the maximum across the *entire* temporal sequence AND all variables. A highly destructive method that loses a lot of specific information.
* **`*_over_patches`**: Compresses the temporal dimension (the patches) but preserves the independence of our 6 variables. Great for capturing the overall behavior of each individual sensor.
* **`*_over_variables`**: Compresses the 6 variables together at each time step. Great for preserving the chronological order of events across the sequence.

In [ ]:
results_dfs = {
    model_name: pd.DataFrame(index=pooling_methods, columns=PATCH_SIZES)
    for model_name in models.keys()
}
for df in results_dfs.values():
    df.index.name = "Pooling \\ Patch"

In [ ]:
for patch in PATCH_SIZES:
    Z_train = Z_train_patch[patch].to(DEVICE)
    Z_test = Z_test_patch[patch].to(DEVICE)
    
    for pool in pooling_methods:
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_pool)
        X_test_scaled = scaler.transform(X_test_pool)
        
        for model_name, clf in models.items():
            clf.fit(X_train_scaled, y_train)
            
            y_pred = clf.predict(X_test_scaled)
            acc = accuracy_score(y_test, y_pred)
            
            results_dfs[model_name].loc[pool, patch] = round(acc, 3)


/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.96243e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.99794e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=2.95261e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditio

In [ ]:
for model_name, df in results_dfs.items():
    print(f"RÉSULTATS POUR : {model_name.upper()}")
    display(df.astype(float))
    
    best_acc = df.max().max()
    best_pool, best_patch = df.astype(float).stack().idxmax()
    print(f"Meilleure combinaison = Patch: {best_patch} | Pooling: '{best_pool}' | Acc: {best_acc:.3f}\n")

RÉSULTATS POUR : RIDGE


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.608,0.625,0.620,0.633
global_mean,0.586,0.590,0.583,0.598
global_max,0.544,0.552,0.536,0.578
global_min,0.545,0.553,0.538,0.565
global_mean_max_min,0.577,0.581,0.577,0.594
mean_over_patches,0.617,0.623,0.619,0.636
max_over_patches,0.590,0.610,0.601,0.632
min_over_patches,0.589,0.605,0.603,0.632
mean_max_min_over_patches,0.611,0.631,0.626,0.633


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.636

RÉSULTATS POUR : RANDOMFOREST


,8,16,32,64
Pooling \ Patch,,,,
flatten,0.567,0.553,0.551,0.592
global_mean,0.574,0.576,0.559,0.579
global_max,0.546,0.522,0.521,0.564
global_min,0.523,0.528,0.515,0.565
global_mean_max_min,0.571,0.573,0.547,0.580
mean_over_patches,0.591,0.578,0.559,0.602
max_over_patches,0.565,0.566,0.552,0.597
min_over_patches,0.559,0.559,0.552,0.597
mean_max_min_over_patches,0.581,0.573,0.564,0.599


Meilleure combinaison = Patch: 64 | Pooling: 'mean_over_patches' | Acc: 0.602



## 3. Multi-Scale Evaluation (Concatenation)

In this section, we concatenate the embeddings from the 4 patch sizes (64+32+16+8) and apply our pooling strategies. This allows the Ridge and Random Forest models to simultaneously access a macro and micro view of the time series.

In [ ]:
df_multiscale_results = pd.DataFrame(index=pooling_methods, columns=models.keys())
df_multiscale_results.index.name = "Pooling Method"

for pool in pooling_methods:
    X_train_multi = []
    X_test_multi = []
    
    for patch in PATCH_SIZES:
        Z_train = Z_train_patch[patch].to(DEVICE)
        Z_test = Z_test_patch[patch].to(DEVICE)
        
        X_train_pool = apply_pooling_pt(Z_train, pool).cpu().numpy()
        X_test_pool = apply_pooling_pt(Z_test, pool).cpu().numpy()
        
        X_train_multi.append(X_train_pool)
        X_test_multi.append(X_test_pool)
        
    X_train_combined = np.concatenate(X_train_multi, axis=1)
    X_test_combined = np.concatenate(X_test_multi, axis=1)
    
    scaler = StandardScaler()
    X_train_combined_scaled = scaler.fit_transform(X_train_combined)
    X_test_combined_scaled = scaler.transform(X_test_combined)

    for model_name, clf in models.items():
        clf.fit(X_train_combined_scaled, y_train)
        
        y_pred = clf.predict(X_test_combined_scaled)
        acc = accuracy_score(y_test, y_pred)
        
        df_multiscale_results.loc[pool, model_name] = round(acc, 3)

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.40241e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.42041e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=1.37498e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:228: LinAlgWarning: Ill-conditio

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.81234e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.86106e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditioned matrix (rcond=3.76852e-08): result may not be accurate.
  dual_coef = linalg.solve(K, y, assume_a="pos", overwrite_a=False)
/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/sklearn/linear_model/_ridge.py:265: LinAlgWarning: Ill-conditio

In [ ]:
display(df_multiscale_results.astype(float))

best_acc = df_multiscale_results.max().max()
best_pool = df_multiscale_results.max(axis=1).idxmax()
best_model = df_multiscale_results.max(axis=0).idxmax()

print("MEILLEURE COMBINAISON MULTI-ÉCHELLE :")
print(f"Modèle : '{best_model}' | Pooling : '{best_pool}'")
print(f"Précision maximale : {best_acc:.3f}")

,Ridge,RandomForest
Pooling Method,,
flatten,0.649,0.580
global_mean,0.623,0.594
global_max,0.591,0.564
global_min,0.587,0.562
global_mean_max_min,0.612,0.590
mean_over_patches,0.658,0.608
max_over_patches,0.659,0.591
min_over_patches,0.650,0.596
mean_max_min_over_patches,0.664,0.599


MEILLEURE COMBINAISON MULTI-ÉCHELLE :
Modèle : 'Ridge' | Pooling : 'mean_max_min_over_patches'
Précision maximale : 0.664
